# SDG&E Microgrid PSPS RL — Step-by-Step Walkthrough

Run this notebook on **CSU TIDE JupyterHub** with the **PyTorch notebook** image.

**Recommended session shape:** 8 CPU · 32 GB RAM · **1× NVIDIA L40** (not A100).

This notebook walks the full pipeline:

1. Install / path setup  
2. GPU check  
3. Generate synthetic data  
4. Inspect the Gymnasium env  
5. Run rule baselines  
6. Smoke-train PPO + SAC (short)  
7. Evaluate + plots  

Full 1e6-step × 5-seed training should use `tide/launch_*_seed.sh` as separate batch jobs (see final section).


## Step 0 — Project root

If you opened the notebook from `notebooks/`, this cell moves the working directory to the repo root.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve repo root whether the kernel cwd is repo root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find pyproject.toml — open this notebook from the project tree.")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
print("ROOT =", ROOT)
print("cwd  =", Path.cwd())


## Step 1 — Install the package

On a fresh TIDE `pytorch-notebook` session, run once. Re-run is safe.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
print("Installed sdge-microgrid editable package.")


## Step 2 — Config + device knobs

- `SMOKE = True` → short training for interactive demos  
- On TIDE with an L40, leave `FORCE_CUDA = True`  
- Locally without a GPU, set `FORCE_CUDA = False`


In [ ]:
from pathlib import Path
import yaml
from sdge_microgrid.device import cuda_summary, resolve_device

SMOKE = True          # True: short timesteps for notebook demos
FORCE_CUDA = True     # True on TIDE L40; False for local CPU-only
SEED = 0
DATA_DIR = Path("data/processed")
RESULTS_DIR = Path("results")
CONFIG_BASE = Path("configs/default.yaml")
CONFIG_TIDE = Path("configs/tide_l40.yaml")

cfg = yaml.safe_load(CONFIG_BASE.read_text())
print("train_timesteps (full protocol):", cfg["train_timesteps"])
print("device (config):", cfg["device"])
print("tide gpu_type:", cfg["tide"]["gpu_type"])

requested = "cuda" if FORCE_CUDA else "cpu"
DEVICE = resolve_device(requested)
print("resolved device:", DEVICE)
print("cuda summary:", cuda_summary())

if FORCE_CUDA and DEVICE != "cuda":
    raise RuntimeError(
        "FORCE_CUDA=True but CUDA is unavailable. "
        "On TIDE: start pytorch-notebook with 1x L40, or set FORCE_CUDA=False for CPU smoke."
    )


## Step 3 — Generate synthetic San Diego traces

Writes `pv`, `pcrit`, `pflex`, `rho_normal`, `rho_highrisk` (8760 hours each) under `data/processed/`.


In [ ]:
from sdge_microgrid.data.synthetic import generate_year_traces, save_processed
import numpy as np

traces = generate_year_traces(seed=42)
save_processed(DATA_DIR, traces)
for k, v in traces.items():
    print(f"{k:14s} shape={v.shape} min={v.min():.3f} max={v.max():.3f} mean={v.mean():.3f}")


## Step 4 — Peek at one week (normal vs high-risk ρ)


In [ ]:
import matplotlib.pyplot as plt
from sdge_microgrid.data.loader import load_week

week_n = load_week(DATA_DIR, start_hour=100, scenario="normal")
week_h = load_week(DATA_DIR, start_hour=2000, scenario="highrisk")

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
t = np.arange(168)
axes[0].plot(t, week_n.pv, label="normal week PV")
axes[0].plot(t, week_h.pv, label="high-risk week PV", alpha=0.7)
axes[0].set_ylabel("PV (kW)")
axes[0].legend()
axes[1].plot(t, week_n.pcrit, label="Pcrit")
axes[1].plot(t, week_n.pflex, label="Pflex")
axes[1].set_ylabel("Load (kW)")
axes[1].legend()
axes[2].plot(t, week_n.rho, label="rho normal")
axes[2].plot(t, week_h.rho, label="rho highrisk")
axes[2].set_ylabel("rho")
axes[2].set_xlabel("Hour in episode")
axes[2].legend()
fig.suptitle("Synthetic week traces")
fig.tight_layout()
plt.show()


## Step 5 — Create `SDGEMicrogridPSPS-v0` and inspect one step

Observation order: `[SoC, Pcrit, Pflex, Ppv, cbuy, g, rho, sin_h, cos_h]`  
Action: `[u_batt ∈ [-1,1], u_shed ∈ [0,1]]`


In [ ]:
import gymnasium as gym
import numpy as np
import sdge_microgrid.env.register  # noqa: F401
from sdge_microgrid.constants import ENV_ID, OBS_DIM

env = gym.make(ENV_ID, data_dir=str(DATA_DIR), start_hour=100, scenario="normal")
obs, info = env.reset(seed=SEED)
print("obs shape:", obs.shape, "expected", (OBS_DIM,))
print("obs:", np.round(obs, 4))
print("info:", info)
print("action_space:", env.action_space)

action = np.array([0.0, 0.0], dtype=np.float32)
obs2, reward, terminated, truncated, info2 = env.step(action)
print("reward:", round(reward, 4))
print("step info keys:", sorted(info2.keys()))
env.close()


## Step 6 — Run rule baselines for one episode each

- **Economic:** TOU arbitrage, ignores ρ  
- **Conservative:** holds reserve when ρ ≥ 0.25


In [ ]:
from sdge_microgrid.baselines import economic_action, conservative_action
from sdge_microgrid.eval.runner import run_episode

def summarize(name, result):
    print(
        f"{name:14s} return={result.episode_return:8.2f}  "
        f"survival={result.critical_survival_rate:6.2f}%  "
        f"cost=${result.weekly_net_economic_cost:7.2f}  "
        f"shed={result.flexible_curtailment_ratio:5.2f}%"
    )

env = gym.make(ENV_ID, data_dir=str(DATA_DIR))
for scenario in ("normal", "highrisk"):
    print(f"\n=== scenario={scenario} start_hour=2000 ===")
    opts = {"start_hour": 2000, "scenario": scenario}
    summarize("economic", run_episode(env, economic_action, seed=SEED, options=opts))
    summarize("conservative", run_episode(env, conservative_action, seed=SEED, options=opts))
env.close()


## Step 7 — Smoke-train PPO

Uses Stable-Baselines3 on the resolved device. With `SMOKE=True`, timesteps are short so the cell finishes in a notebook session.

For the full protocol (`train_timesteps` from config, usually 1e6), use the TIDE launch scripts in the last section instead.


In [ ]:
from _common import load_config, make_vec_env
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback

overlay = CONFIG_TIDE if DEVICE == "cuda" else None
train_cfg = load_config(CONFIG_BASE, overlay)
timesteps = 4096 if SMOKE else int(train_cfg["train_timesteps"])
out = RESULTS_DIR / "ppo" / f"seed_{SEED}"
out.mkdir(parents=True, exist_ok=True)

train_env = make_vec_env(DATA_DIR, SEED)
eval_env = make_vec_env(DATA_DIR, SEED + 10_000)
model_ppo = PPO(
    "MlpPolicy",
    train_env,
    device=DEVICE,
    gamma=float(train_cfg["gamma"]),
    seed=SEED,
    verbose=1,
    **train_cfg["ppo"],
)
callback = EvalCallback(
    eval_env,
    best_model_save_path=str(out),
    log_path=str(out),
    eval_freq=2048 if SMOKE else 10_000,
    deterministic=True,
)
model_ppo.learn(total_timesteps=timesteps, callback=callback)
model_ppo.save(str(out / "final_model"))
print(f"PPO saved to {out} (device={DEVICE}, timesteps={timesteps})")


## Step 8 — Smoke-train SAC


In [ ]:
from stable_baselines3 import SAC

timesteps_sac = 3000 if SMOKE else int(train_cfg["train_timesteps"])
out_sac = RESULTS_DIR / "sac" / f"seed_{SEED}"
out_sac.mkdir(parents=True, exist_ok=True)

train_env = make_vec_env(DATA_DIR, SEED)
eval_env = make_vec_env(DATA_DIR, SEED + 10_000)
model_sac = SAC(
    "MlpPolicy",
    train_env,
    device=DEVICE,
    gamma=float(train_cfg["gamma"]),
    seed=SEED,
    verbose=1,
    **train_cfg["sac"],
)
callback = EvalCallback(
    eval_env,
    best_model_save_path=str(out_sac),
    log_path=str(out_sac),
    eval_freq=1500 if SMOKE else 10_000,
    deterministic=True,
)
model_sac.learn(total_timesteps=timesteps_sac, callback=callback)
model_sac.save(str(out_sac / "final_model"))
print(f"SAC saved to {out_sac} (device={DEVICE}, timesteps={timesteps_sac})")


## Step 9 — Quick frozen evaluation (baselines + trained policies)

Runs the held-out week lists for **seed 0** only (faster). Use `scripts/evaluate_all.py --seeds 0,1,2,3,4` after full multi-seed training.


In [ ]:
from pathlib import Path
import csv
from sdge_microgrid.eval.sb3_policy import make_sb3_policy
from _common import ALGOS, model_path

NORMAL_STARTS = [100, 400, 800, 1200, 1600]  # subset for notebook speed
HIGHRISK_STARTS = [2000, 2040, 4500, 4540, 7000]

rows = []
env = gym.make(ENV_ID, data_dir=str(DATA_DIR))

def eval_policy(name, policy):
    for scenario, starts in (("normal", NORMAL_STARTS), ("highrisk", HIGHRISK_STARTS)):
        for start in starts:
            res = run_episode(
                env,
                policy,
                seed=SEED * 10_000 + start,
                options={"start_hour": start, "scenario": scenario},
            )
            rows.append({
                "controller": name,
                "seed": SEED,
                "scenario": scenario,
                "start_hour": start,
                "critical_survival_rate": res.critical_survival_rate,
                "weekly_net_economic_cost": res.weekly_net_economic_cost,
                "flexible_curtailment_ratio": res.flexible_curtailment_ratio,
                "episode_return": res.episode_return,
            })

eval_policy("economic", economic_action)
eval_policy("conservative", conservative_action)

for algo in ("ppo", "sac"):
    path = model_path(RESULTS_DIR, algo, SEED)
    if path.exists():
        model = ALGOS[algo].load(str(path), device=DEVICE)
        eval_policy(algo, make_sb3_policy(model))
    else:
        print(f"skip {algo}: missing {path}")

env.close()

table_dir = RESULTS_DIR / "tables"
table_dir.mkdir(parents=True, exist_ok=True)
csv_path = table_dir / "comparison_notebook.csv"
with csv_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

import pandas as pd
df = pd.DataFrame(rows)
summary = df.groupby(["controller", "scenario"])[
    ["critical_survival_rate", "weekly_net_economic_cost", "flexible_curtailment_ratio"]
].mean().round(3)
print(summary)
print("\nWrote", csv_path)


## Step 10 — 72-hour trajectory plot (high-risk week)


In [ ]:
from sdge_microgrid.eval.plots import plot_trajectory_72h

env = gym.make(ENV_ID, data_dir=str(DATA_DIR))
policy = conservative_action
ppo_path = model_path(RESULTS_DIR, "ppo", SEED)
if ppo_path.exists():
    policy = make_sb3_policy(ALGOS["ppo"].load(str(ppo_path), device=DEVICE))

res = run_episode(
    env,
    policy,
    seed=SEED,
    options={"start_hour": 1980, "scenario": "highrisk"},
)
fig_dir = RESULTS_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
out_png = fig_dir / "trajectory_72h_notebook.png"
plot_trajectory_72h(res.infos[:72], out_png, title="72h dispatch (high-risk, notebook)")
env.close()

from IPython.display import Image, display
display(Image(filename=str(out_png)))
print("survival=", round(res.critical_survival_rate, 2),
      "cost=", round(res.weekly_net_economic_cost, 2),
      "shed=", round(res.flexible_curtailment_ratio, 2))


## Step 11 — Optional: unit tests

Skip if you only want the RL walkthrough.


In [ ]:
import subprocess, sys
# Fast subset; full suite: subprocess.check_call([sys.executable, "-m", "pytest", "-q"])
subprocess.check_call([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_constants.py",
    "tests/test_device.py",
    "tests/test_env.py",
    "tests/test_baselines.py",
])


## Step 12 — Full TIDE multi-seed training (batch, not notebook)

When `SMOKE` results look sane, launch the protocol outside this interactive session:

```bash
# Prefer one TIDE batch container job per line (1x L40 each):
bash tide/launch_ppo_seed.sh 0
bash tide/launch_ppo_seed.sh 1
bash tide/launch_ppo_seed.sh 2
bash tide/launch_ppo_seed.sh 3
bash tide/launch_ppo_seed.sh 4
bash tide/launch_sac_seed.sh 0
# ... seeds 1-4 for SAC

# After all 10 jobs finish:
PYTHONPATH=src python scripts/evaluate_all.py --seeds 0,1,2,3,4
PYTHONPATH=src python scripts/plot_results.py
PYTHONPATH=src python scripts/check_hypotheses.py
```

Hypothesis targets (proposal): critical survival **> 98%** in outages; normal-week bill **> 15%** below conservative.

> Smoke numbers from this notebook are **not** experimental findings — regenerate with the full timestep budget on TIDE before reporting.
